In [86]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=2"

from jVMC_exp.sharding_config import MESH, DEVICE_SHARDING
import jax
import jax.numpy as jnp

print("Devices =", jax.devices())
print("Device count =", jax.device_count())
print("Mesh shape =", MESH.shape["devices"])

Devices = [CpuDevice(id=0), CpuDevice(id=1)]
Device count = 2
Mesh shape = 2


In [5]:
import jax
# jax.config.update("jax_log_compiles", True)
import jax.numpy as jnp

from jVMC_exp.vqs import NQS
from jVMC_exp.sharding_config import sharded
import jVMC_exp


In [6]:
class TestNQS(NQS):
    @sharded(automatic_sharding=True, yield_iter=True)
    def _gradients_iter_sh(self, s, *, parameters, batch_size):
        return self.flat_gradient_function(self.apply_fun, parameters, s)

    def iterative_gradients(self, s):
        return self._gradients_iter_sh(s, parameters=self.grad_parameters, batch_size=self.batchSize)

In [7]:
L = 10
n_samples = 2**8
n_chains = 2**6
batch_size = 2**4

model = jVMC_exp.nets.RBM(L)
psi = NQS(model, L, batch_size, seed=123)
sampler = jVMC_exp.sampler.MCSampler(
    psi, 
    jVMC_exp.propose.SpinFlip(),
    123, 
    n_chains,
    n_samples    
)

samples = sampler.samples

In [36]:
iterable = iter(psi.iterative_gradients(samples))

In [46]:
next(iterable)

Array([[-0.16447629+0.j, -0.19154633+0.j, -0.46060452+0.j, ...,
         0.70321061+0.j,  0.61409949+0.j,  0.57386603+0.j],
       [ 0.52780688+0.j,  0.00920603+0.j, -0.98801069+0.j, ...,
         0.87786716+0.j, -0.37917495+0.j, -0.69983218+0.j],
       [ 0.66875099+0.j,  0.64812266+0.j,  0.09611001+0.j, ...,
         0.93837678+0.j,  0.84726851+0.j, -0.96734521+0.j],
       ...,
       [ 0.91607644+0.j,  0.90918861+0.j, -0.1580211 +0.j, ...,
         0.84259112+0.j,  0.87658272+0.j, -0.97292724+0.j],
       [ 0.93174157+0.j,  0.55527021+0.j, -0.97245391+0.j, ...,
         0.7625902 +0.j, -0.61590468+0.j, -0.9010704 +0.j],
       [ 0.77987809+0.j,  0.12308084+0.j, -0.87995221+0.j, ...,
         0.7734512 +0.j,  0.33002167+0.j, -0.56394457+0.j]],      dtype=complex128)

In [7]:
gradients = []
iterable = psi.iterative_gradients(samples)

for grad in iterable:
    gradients.append(grad)
gradients = jnp.concatenate(gradients)

jnp.allclose(gradients, psi.gradients(samples))

Array(True, dtype=bool)